# Pipeline Report

In [ ]:
import json
import pandas as pd
from src import config
from pathlib import Path
from IPython.display import display, Markdown

## Data Profile

In [ ]:
records = []

for file_path in sorted(config.METRICS_DIR.glob("*_understanding.json")):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    target = data.get("target_col", "close")
    stats = data.get("stats", {}).get(target, {})

    records.append({
        "Ticker": data.get("ticker"),
        "Periode": f"{data.get('min_date')} - {data.get('max_date')}",
        "N": data.get("total_rows"),
        "Min": stats.get("min"),
        "Q1 (25%)": stats.get("25%"),
        "Median (50%)": stats.get("50%"),
        "Q3 (75%)": stats.get("75%"),
        "Max": stats.get("max"),
        "Mean": stats.get("mean"),
        "Std Dev": stats.get("std"),
    })

df = pd.DataFrame(records).reset_index(drop=True)
display(Markdown(df.to_markdown(index=False)))


## Evaluation

In [ ]:
eval_records = []

for file_path in sorted(config.METRICS_DIR.glob("*_evaluation.json")):
    with open(file_path, "r", encoding="utf-8") as f:
        eval_data = json.load(f)

    ticker = file_path.stem.replace("_evaluation", "")

    for item in eval_data:
        eval_records.append({
            "Ticker": ticker,
            "Model": item.get("model"),
            "Time Steps": item.get("time_steps"),
            "Batch Size": item.get("batch_size"),
            "Learning Rate": item.get("learning_rate"),
            "MSE": item.get("MSE"),
            "RMSE": item.get("RMSE"),
            "MAPE": item.get("MAPE"),
        })

df_eval = pd.DataFrame(eval_records).reset_index(drop=True)
display(Markdown(df_eval.to_markdown(index=False)))


## Visualisasi Hasil Inferensi

In [ ]:
plot_files = sorted(config.PLOTS_DIR.glob("*_inference.png"))

for plot_file in plot_files:
    filename = plot_file.name
    parts = filename.replace("_inference.png", "").split("_")
    ticker = parts[0]
    model_name = parts[1] if len(parts) > 1 else ""
    title = f"{ticker} ({model_name})" if model_name else ticker

    display(Markdown(f"![{title}](artifact/plots/{filename})\n"))